# Install Libraries

In [1]:
!pip install -q -U transformers accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 87.9 MB/s eta 0:00:00


# Import Libraries

In [2]:
import torch
import json
import random

from transformers import AutoTokenizer, AutoModelForCausalLM

# Set Model Path

In [3]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    if "model.safetensors" in files:
        print(root)

/kaggle/input/models/lama377/midad-allam-7b/transformers/default/1/merged_full_model


In [4]:
MODEL_DIR = "/kaggle/input/models/lama377/midad-allam-7b/transformers/default/1/merged_full_model"

print("Model path:", MODEL_DIR)

Model path: /kaggle/input/models/lama377/midad-allam-7b/transformers/default/1/merged_full_model


# Load Tokenizer and Model

In [5]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("✅ Midad model loaded successfully")
print("Device:", model.device)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Midad model loaded successfully
Device: cuda:0


# Write a New Lesson

In [6]:
new_lesson = """
عنوان الدرس:
دورة الماء

أهداف التعلم:
* أن يشرح الطفل مراحل دورة الماء.
* أن يفرق الطفل بين التبخر والتكاثف.
* أن يتعرف الطفل على الهطول.

المفاهيم الأساسية:
* التبخر
* التكاثف
* الهطول

المحتوى التعليمي:
تتحرك المياه في الطبيعة في دورة مستمرة.
تسخن الشمس الماء فيتبخر ويرتفع إلى الهواء.
ثم يبرد بخار الماء ويتكاثف مكوناً السحب.
وعندما تصبح قطرات الماء ثقيلة فإنها تسقط على شكل مطر أو أنواع أخرى من الهطول.

المهمة:
حوّل هذا الدرس إلى قصة تعليمية ممتعة ومناسبة لطفل سعودي عمره من 7 إلى 10 سنوات.
"""

# Generate Story for New Lesson

In [7]:
import torch

# =========================================================
# System Prompt
# =========================================================

improved_system_prompt = (
    "أنت كاتب قصص تعليمية للأطفال من عمر 7 إلى 10 سنوات. "
    "مهمتك تحويل الدرس المعطى إلى قصة تعليمية ممتعة وواضحة ومناسبة لعمر الطفل، "
    "مع المحافظة على دقة المعلومات الواردة في الدرس.\n\n"

    "قواعد مهمة:\n"

    "1) التزم بالمحتوى التعليمي وأهداف التعلم والمفاهيم الأساسية الموجودة في الدرس.\n"

    "2) لا تضف معلومات علمية أو رياضية أو صحية غير متأكد من صحتها.\n"

    "3) حاول تغطية جميع أهداف التعلم والمفاهيم الأساسية داخل أحداث القصة.\n"

    "4) إذا كان الدرس يحتوي على أرقام أو عمليات حسابية، "
    "تأكد من صحة النتائج قبل كتابتها.\n"

    "5) لا تخلط بين المفاهيم المختلفة أو المتشابهة.\n"

    "6) استخدم أمثلة بسيطة وواضحة ومناسبة للأطفال.\n"

    "7) أضف لمسات طبيعية من البيئة والثقافة السعودية، "
    "وعند ذكر مكان استخدم مدينة أو منطقة سعودية حقيقية بشكل طبيعي.\n"

    "8) اجعل القصة مترابطة من البداية للنهاية.\n"

    "9) أخرج القصة فقط دون ملاحظات أو شرح أو ملخص بعد نهايتها.\n"

    "10) لا تكتب رموزاً أو قوالب مثل "
    "[INST] أو [/INST] أو <<SYS>> أو </SYS> أو [END_TEXT] أو روابط.\n"

    "11) عندما تنتهي القصة، توقف مباشرة."
)

# =========================================================
# Build Messages
# =========================================================

messages = [
    {
        "role": "system",
        "content": improved_system_prompt
    },
    {
        "role": "user",
        "content": new_lesson
    }
]

# =========================================================
# Tokenize
# =========================================================

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

# =========================================================
# Generate Story
# =========================================================

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False,
        repetition_penalty=1.12,
        no_repeat_ngram_size=4,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

# =========================================================
# Decode Generated Story
# =========================================================

generated_story = tokenizer.decode(
    out[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
).strip()

print("===== القصة المولّدة =====\n")
print(generated_story)

===== القصة المولّدة =====

في رحلة مدرسية إلى شاطئ نصف القمر بالدمام، لاحظت المعلمة هند أن البحر يبدو هادئاً جداً اليوم. سألت الطلاب: «هل تعرفون لماذا؟» أجاب أحد الطلاب: «ربما لأن الجو حار الآن». قالت المعلمة هند: «صحيح تماماً! الحرارة تجعل ماء البحر يتبخر».

سأل طالب آخر: «وما هو التبخر يا أستاذة؟» أجابت المعلمة: «التبخر هو تحول الماء من حالته السائلة إلى بخار يرتفع في الهواء بسبب حرارة الشمس». أضافت: «وهكذا يتحرك الماء باستمرار في دورة تسمى دورة الماء في الطبيعة».

بعد قليل، لاحظ الطلاب سحابة بيضاء كبيرة تتشكل فوقهم. سألت المعلمة: «ماذا تلاحظون الآن؟» أجاب أحدهم: «السحابة تكبر وتكبر!» قالت المعلمة: «هذا صحيح، البخار الذي تبخر من البحر ارتفع ووصل إلى السماء، وعندما برد أصبح تكاثفاً، أي تحول مرة أخرى إلى قطرات صغيرة تشكلت معاً لتكوّن السحابة».

قال طالب آخر:«إذن، هذه القطرات الصغيرة هي التي ستتحول لاحقاً إلى مطر؟» ابتسمت المعلمة وقالت: «بالضبط! وهذا ما يسمى بالهطول، وهو المرحلة الأخيرة من دورة الماء حيث يسقط المطر من السحاب ويعود إلى الأرض ليغذي البحار والمحيطات مرة أخرى».

في نهاية ا

# Load Test Data

In [8]:
DATA_DIR = "/kaggle/input/datasets/lama377/midad-training-data"
TEST_PATH = f"{DATA_DIR}/merged_test.jsonl"

def read_jsonl(path):
    rows = []

    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                rows.append(json.loads(line))

    return rows

test_rows = read_jsonl(TEST_PATH)

print("عدد أمثلة test:", len(test_rows))

عدد أمثلة test: 64


# Generate Stories for 8 Test Lessons

In [9]:
import random
import torch

# =========================================================
# System Prompt أقوى للدقة التعليمية
# =========================================================

improved_system_prompt = (
    "أنت كاتب قصص تعليمية للأطفال من عمر 7 إلى 10 سنوات. "
    "مهمتك تحويل الدرس المعطى إلى قصة تعليمية ممتعة وواضحة ومناسبة لعمر الطفل، "
    "مع المحافظة الصارمة على دقة المعلومات الواردة في الدرس.\n\n"

    "قواعد إلزامية يجب اتباعها:\n"

    "1) اعتبر المحتوى التعليمي وأهداف التعلم والمفاهيم الأساسية في رسالة المستخدم "
    "هي المصدر الوحيد للمعلومات التعليمية في القصة.\n"

    "2) لا تضف أي معلومة علمية أو رياضية أو صحية أو لغوية جديدة غير مذكورة في الدرس "
    "إذا كان من الممكن أن تكون غير دقيقة.\n"

    "3) يجب أن تغطي القصة جميع أهداف التعلم وجميع المفاهيم الأساسية المذكورة في الدرس. "
    "لا تتجاهل أي هدف أو مفهوم.\n"

    "4) عند وجود أرقام أو عمليات حسابية، استخدم فقط عمليات صحيحة وتحقق من الناتج قبل كتابته.\n"

    "5) عند وجود تعريفات أو خصائص، التزم بالنص التعليمي كما ورد، "
    "ولا تخترع خصائص إضافية.\n"

    "6) عند ذكر أمثلة، اختر أمثلة بسيطة وواضحة ومتوافقة مباشرة مع المفاهيم المذكورة في الدرس. "
    "إذا لم تكن متأكدًا من صحة المثال، فلا تذكره.\n"

    "7) لا تخلط بين المفاهيم المتشابهة. "
    "مثلاً: لا تخلط بين الدائرة والكرة، أو بين المربع والمستطيل، "
    "أو بين التبخر والتكاثف، أو بين مواضع الحروف في الكلمات.\n"

    "8) إذا كان الدرس لغويًا، فتأكد أن الكلمات التي تستخدمها تحقق فعلاً الحرف أو الموضع المطلوب.\n"

    "9) إذا كان الدرس رياضيًا، يجب أن تكون جميع الأمثلة العددية صحيحة تمامًا.\n"

    "10) إذا كان الدرس علميًا أو صحيًا، لا تدّعِ نتائج علاجية أو معلومات إضافية "
    "غير موجودة في النص التعليمي.\n"

    "11) أضف لمسات طبيعية من البيئة والثقافة السعودية. "
    "وعند ذكر مكان القصة، استخدم اسم مدينة أو منطقة سعودية حقيقية ومميزة بشكل طبيعي.\n"

    "12) اجعل القصة مترابطة، ولا تنتقل إلى قصة ثانية أو موضوع مختلف.\n"

    "13) أخرج القصة فقط. "
    "لا تكتب ملاحظات، ولا شرحًا، ولا ملخصًا، ولا تعليقًا على جودة القصة.\n"

    "14) لا تكتب أي رموز أو قوالب مثل "
    "[INST] أو [/INST] أو <<SYS>> أو </SYS> أو [END_TEXT] أو روابط أو وسوم برمجية.\n"

    "15) عندما تنتهي القصة، توقف مباشرة.\n"

    "قبل كتابة الإجابة النهائية، راجع داخليًا ما يلي:\n"
    "- هل غطيت كل أهداف التعلم؟\n"
    "- هل غطيت كل المفاهيم الأساسية؟\n"
    "- هل كل الأرقام والعمليات صحيحة؟\n"
    "- هل كل الأمثلة متوافقة مع محتوى الدرس؟\n"
    "- هل أضفت أي معلومة غير موجودة في الدرس قد تكون غير دقيقة؟ إذا نعم، احذفها."
)

# =========================================================
# اختيار نفس 8 أمثلة
# =========================================================

random.seed(42)

sample_size = 8
test_sample = random.sample(test_rows, sample_size)

results = []

# =========================================================
# توليد القصص
# =========================================================

for i, example in enumerate(test_sample):

    user_message = [
        m for m in example["messages"]
        if m["role"] == "user"
    ][0]

    messages_input = [
        {
            "role": "system",
            "content": improved_system_prompt
        },
        {
            "role": "user",
            "content": user_message["content"]
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages_input,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=False,
            repetition_penalty=1.12,
            no_repeat_ngram_size=4,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = tokenizer.decode(
        out[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    # تنظيف احتياطي
    bad_markers = [
        "[INST]",
        "[/INST]",
        "<<SYS>>",
        "<</SYS>>",
        "</SYS>",
        "[END_TEXT]",
        "[/END_TEXT]",
        "[END_OF_TEXT]",
        "[/END_OF_TEXT]",
        "<![CDATA[",
        "CDATA",
    ]

    for marker in bad_markers:
        if marker in generated:
            generated = generated.split(marker)[0].strip()

    results.append({
        "index": i + 1,
        "user_prompt": user_message["content"],
        "generated_story": generated,
    })

    print(f"\n===== مثال {i+1}/{sample_size} =====")
    print("\nالدرس:")
    print(user_message["content"])
    print("\nالقصة المولّدة:")
    print(generated)
    print("\n" + "=" * 100)


===== مثال 1/8 =====

الدرس:
عنوان الدرس:
الأعداد من ١ إلى ١٠

أهداف التعلم:
* أن يعد الطفل الأشياء من ١ إلى ١٠.
* أن يتعرف الطفل على شكل الأرقام من ١ إلى ١٠.
* أن يرتب الطفل الأعداد تصاعدياً وتنازلياً ضمن العشرة.

المفاهيم الأساسية:
* العد التصاعدي
* العد التنازلي
* الأرقام من ١ إلى ١٠
* ترتيب الأعداد

المحتوى التعليمي:
العد التصاعدي يعني العد من الأصغر للأكبر (١، ٢، ٣...١٠)، والعد التنازلي عكسه (١٠، ٩، ٨...١). كل رقم من ١ إلى ١٠ له شكل مختلف نتعرف عليه ونربطه بعدد الأشياء الذي يمثله.

المهمة:
حوّل هذا الدرس إلى قصة تعليمية ممتعة ومناسبة لطفل سعودي عمره من 7 إلى 10 سنوات، مع المحافظة على دقة المعلومات التعليمية.

القصة المولّدة:
في رحلة مدرسية إلى حديقة الملك عبدالله بالرياض، طلبت المعلمة هند من الطلاب أن يعدوا الألعاب الموجودة حولهم. وقف الطالب فيصل أمام مجموعة ألعاب ملونة وقال: "هذه خمسة ألعاب يا أستاذة". عدّتها المعلمة معه وقالت: "صحيح، أنت جيد في العد".

بعد قليل، وقفت الطالبة نورة بجانب طاولة صغيرة عليها عشر كرات زجاجية ملونة بألوان مختلفة. قالت: "انظروا، هذه الكرات عددها عشرة"،

In [10]:
import json


# Save Test Results

In [11]:

RESULTS_PATH = "/kaggle/working/midad_test_results.json"

with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2
    )

print("✅ Results saved to:")
print(RESULTS_PATH)

✅ Results saved to:
/kaggle/working/midad_test_results.json
